# 1. EDA

**Course:** TC5035.10 - Capstone project
**Tecnológico de Monterrey**

**Professor:** Raúl Valente Ramírez Velarde

---
**Team:** 57
> César Miguel Barrientos Robles - A01796615
>
> Alan Julian Rodríguez García - A01796833

## Context
According to Mexico's 2020 Population and Housing Census (INEGI), Mexico registered 6.1 million people with some form of disability,
21.9% of whom — approximately 1.35 million — are deaf or hard of hearing. For many of them, Mexican Sign Language (LSM) is their primary means of communication. Yet the vast majority of the hearing population has no exposure to it, creating a persistent communication barrier hat technology has largely failed to address.

LSM has its own vocabulary and a complex grammar that is very different from Spanish grammar — it is a complete, natural language of the Mexican deaf community, capable of expressing as wide a range of thoughts and emotions as any other language.  It is not simply Spanish translated into gestures. Despite being part of Mexico's national linguistic heritage, there are no official statistics on the precise number of LSM speakers or users.

LSM was officially recognized in the General Law for the Inclusion of Persons with Disabilities in 2005.  Since then, some progress has been made — certain news programs and government broadcasts include LSM interpretation — but digital tools built specifically for LSM remain scarce. Most existing sign language recognition systems are built for American Sign Language (ASL) and do not transfer well to LSM.

**manos-hablando** is a first step toward closing that gap. The full vision is a real-time pipeline that captures hand signs, recognizes them using a Transformer-based classifier trained specifically on LSM, and passes the recognized signs to a Large Language Model (LLM) that reconstructs natural Spanish from them — bridging the communication gap in real time.

This notebook documents the first look at the data — before any model sees it.

### EDA and preprocesing

In [19]:
import json
from pathlib import Path

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os

from manos_hablando.config import PROCESSED_DATA_DIR



In [ ]:
LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]

# Build column names: WRIST_x, WRIST_y, WRIST_z, THUMB_CMC_x, ...
columns = []
for name in LANDMARK_NAMES:
    for coord in ["x", "y", "z"]:
        columns.append(f"{name}_{coord}")

# Load and flatten
with open(PROCESSED_DATA_DIR / "keypoints.json") as f:
    data = json.load(f)

rows = []
for entry in data:
    row = {
        "letter": entry["letter"],
        "file": entry["file"],
    }

    # keypoints is nested as a dict
    kp_data = entry["keypoints"]

    # handle handedness if available
    row["handedness"] = kp_data.get("handedness")
    row["handedness_confidence"] = kp_data.get("handedness_confidence")

    # image keypoints
    flat = [coord for point in kp_data["keypoints"] for coord in point]
    row.update(dict(zip(columns, flat)))

    # world keypoints
    if "world_keypoints" in kp_data:
        world_columns = [f"{name}_w{coord}" for name in LANDMARK_NAMES for coord in ["x", "y", "z"]]
        flat_world = [coord for point in kp_data["world_keypoints"] for coord in point]
        row.update(dict(zip(world_columns, flat_world)))

    rows.append(row)

df = pd.DataFrame(rows)
display(df.head())

In [12]:
print("-" * 50)
print("Basic information about the dataset:")
print("-" * 50)
display(df.info())


--------------------------------------------------
Basic information about the dataset:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2057 entries, 0 to 2056
Data columns (total 65 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   letter        2057 non-null   object 
 1   file          2057 non-null   object 
 2   WRIST_x       2057 non-null   float64
 3   WRIST_y       2057 non-null   float64
 4   WRIST_z       2057 non-null   float64
 5   THUMB_CMC_x   2057 non-null   float64
 6   THUMB_CMC_y   2057 non-null   float64
 7   THUMB_CMC_z   2057 non-null   float64
 8   THUMB_MCP_x   2057 non-null   float64
 9   THUMB_MCP_y   2057 non-null   float64
 10  THUMB_MCP_z   2057 non-null   float64
 11  THUMB_IP_x    2057 non-null   float64
 12  THUMB_IP_y    2057 non-null   float64
 13  THUMB_IP_z    2057 non-null   float64
 14  THUMB_TIP_x   2057 non-null   float64
 15  THUMB_TIP_y   2057

None

In [13]:
# Categorical columns
cat_cols = df.select_dtypes(include="object").columns.tolist()

# Numerical columns
num_cols = df.select_dtypes(include="float64").columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

Categorical columns (2): ['letter', 'file']
Numerical columns (63): ['WRIST_x', 'WRIST_y', 'WRIST_z', 'THUMB_CMC_x', 'THUMB_CMC_y', 'THUMB_CMC_z', 'THUMB_MCP_x', 'THUMB_MCP_y', 'THUMB_MCP_z', 'THUMB_IP_x', 'THUMB_IP_y', 'THUMB_IP_z', 'THUMB_TIP_x', 'THUMB_TIP_y', 'THUMB_TIP_z', 'INDEX_MCP_x', 'INDEX_MCP_y', 'INDEX_MCP_z', 'INDEX_PIP_x', 'INDEX_PIP_y', 'INDEX_PIP_z', 'INDEX_DIP_x', 'INDEX_DIP_y', 'INDEX_DIP_z', 'INDEX_TIP_x', 'INDEX_TIP_y', 'INDEX_TIP_z', 'MIDDLE_MCP_x', 'MIDDLE_MCP_y', 'MIDDLE_MCP_z', 'MIDDLE_PIP_x', 'MIDDLE_PIP_y', 'MIDDLE_PIP_z', 'MIDDLE_DIP_x', 'MIDDLE_DIP_y', 'MIDDLE_DIP_z', 'MIDDLE_TIP_x', 'MIDDLE_TIP_y', 'MIDDLE_TIP_z', 'RING_MCP_x', 'RING_MCP_y', 'RING_MCP_z', 'RING_PIP_x', 'RING_PIP_y', 'RING_PIP_z', 'RING_DIP_x', 'RING_DIP_y', 'RING_DIP_z', 'RING_TIP_x', 'RING_TIP_y', 'RING_TIP_z', 'PINKY_MCP_x', 'PINKY_MCP_y', 'PINKY_MCP_z', 'PINKY_PIP_x', 'PINKY_PIP_y', 'PINKY_PIP_z', 'PINKY_DIP_x', 'PINKY_DIP_y', 'PINKY_DIP_z', 'PINKY_TIP_x', 'PINKY_TIP_y', 'PINKY_TIP_z']


In [14]:
# Show the statistics of the numerical columns
df[ num_cols].describe()

,WRIST_x,WRIST_y,WRIST_z,THUMB_CMC_x,THUMB_CMC_y,THUMB_CMC_z,THUMB_MCP_x,THUMB_MCP_y,THUMB_MCP_z,THUMB_IP_x,...,PINKY_MCP_z,PINKY_PIP_x,PINKY_PIP_y,PINKY_PIP_z,PINKY_DIP_x,PINKY_DIP_y,PINKY_DIP_z,PINKY_TIP_x,PINKY_TIP_y,PINKY_TIP_z
count,2057.000000,2057.000000,2.057000e+03,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,...,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000,2057.000000
mean,0.480572,0.650360,-2.897747e-08,0.482694,0.619819,-0.024906,0.484810,0.573021,-0.043031,0.485167,...,-0.065201,0.483784,0.531437,-0.091098,0.482337,0.540420,-0.088409,0.481795,0.547408,-0.081803
std,0.268858,0.194471,6.814772e-07,0.217950,0.169687,0.030560,0.187326,0.150977,0.047992,0.190151,...,0.063969,0.252171,0.172036,0.079867,0.241299,0.175733,0.078633,0.238134,0.182451,0.077541
min,-0.023785,-0.032846,-2.998706e-06,0.055463,0.082844,-0.195425,0.066835,0.084264,-0.362882,0.053669,...,-0.492440,-0.021382,0.080338,-0.555712,-0.017099,0.106224,-0.521632,-0.006633,0.035910,-0.508407
25%,0.250131,0.513076,-3.481090e-07,0.300808,0.494025,-0.038649,0.335539,0.459378,-0.063333,0.334290,...,-0.087390,0.270189,0.408488,-0.124968,0.274213,0.404630,-0.121358,0.275710,0.410741,-0.111202
50%,0.405496,0.668723,2.897328e-08,0.437790,0.624399,-0.018264,0.454505,0.573278,-0.031826,0.465032,...,-0.046000,0.445661,0.519679,-0.069724,0.441192,0.533664,-0.067003,0.443376,0.544362,-0.060487
75%,0.742720,0.802403,2.917660e-07,0.684179,0.748014,-0.006512,0.649056,0.683047,-0.013190,0.640588,...,-0.024864,0.707510,0.655033,-0.037420,0.689081,0.669207,-0.036796,0.690797,0.675926,-0.032159
max,1.058916,1.052532,3.677860e-06,0.966844,1.006467,0.144135,0.904617,0.956149,0.183174,0.953803,...,0.246675,1.043936,1.023387,0.301850,1.035404,1.027916,0.310679,1.036546,1.030586,0.309377


In [15]:
# Show the statistics of the categorical columns
df[cat_cols].describe()

,letter,file
count,2057,2057
unique,26,1209
top,RR,aug_0007.jpg
freq,100,17


#### Description of the dataset
The dataset consists of 2,057 samples across 26 LSM letters, with 63 numerical features
per sample (21 hand landmarks × 3 coordinates) extracted by MediaPipe Hand Landmarker.

**General:**
- No missing values across any column
- 2 categorical columns: `letter` (target) and `file` (identifier)
- 63 numerical columns: `x`, `y`, `z` per landmark

**Class distribution:**
- Balanced at ~100 samples per letter
- Achieved via volunteer images + augmentation
- All 26 letters present

**X and Y coordinates:**
- Mean x ≈ 0.48 — hands consistently centered horizontally
- Mean y ≈ 0.65 — hands tend to sit in the lower half of the frame
- Minor extrapolations beyond 0-1 observed (min: -0.03, max: 1.06) — hand near frame edge

**Z coordinate (depth):**
- Wrist z ≈ 0.0 always — serves as the depth reference point
- Fingertips show the most negative z values (INDEX_TIP mean: -0.103) — closest to camera
- Clear depth gradient from wrist → knuckles → fingertips across all five fingers

**Variability:**
- Std of ~0.19-0.26 across x and y reflects natural variation in hand position across contributors
- This confirms the need for wrist-centered normalization applied during training